# Notebook to build recommneder system
will build a hybrdi system that include collabrative filtering, content based

In [43]:
# loading libraries to deal with tables and numbers
import pandas as pd
import numpy as np

In [44]:
# loading and checking dataset
pd.set_option('display.max_columns', None)
df = pd.read_csv("file.csv")
df.head(3)

,Unnamed: 0,CustomerID,Gender,Location,Tenure_Months,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,GST,Date,Offline_Spend,Online_Spend,Month,Coupon_Code,Discount_pct
0,0,17850.0,M,Chicago,12.0,16679.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1.0,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
1,1,17850.0,M,Chicago,12.0,16680.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1.0,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
2,2,17850.0,M,Chicago,12.0,16696.0,2019-01-01,GGOENEBQ078999,Nest Cam Outdoor Security Camera - USA,Nest-USA,2.0,122.77,6.5,Not Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0


In [45]:
df.shape

(52955, 21)

In [46]:
# seeing how many unique vlaues in some columns to know how to encode them later
uni_cus = df['Product_Description'].nunique()
uni_cou = df['Coupon_Status'].nunique()
uni_cod = df['Coupon_Code'].nunique()
uni_pro = df['Product_Category'].nunique()

print(uni_cus, uni_cou, uni_cod,uni_pro)

404 3 48 21


In [47]:
# adding product id column
df["Produc_ID"] = df["Product_Description"].astype("category").cat.codes.astype("int64")

In [48]:
# drop not needed columns
df = df.drop(["Unnamed: 0", "Transaction_ID", "Product_SKU", 'Date', 'Product_Description', 'Coupon_Code'], axis=1)

In [49]:
# removing null and duplicate values
df = df.dropna()
df = df.drop_duplicates()
df.shape

(48672, 16)

In [50]:
# creating score column
df["score"] = df["Quantity"] * df["Avg_Price"]
df.head(2)

,CustomerID,Gender,Location,Tenure_Months,Transaction_Date,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,GST,Offline_Spend,Online_Spend,Month,Discount_pct,Produc_ID,score
0,17850.0,M,Chicago,12.0,2019-01-01,Nest-USA,1.0,153.71,6.5,Used,0.1,4500.0,2424.5,1,10.0,316,153.71
2,17850.0,M,Chicago,12.0,2019-01-01,Nest-USA,2.0,122.77,6.5,Not Used,0.1,4500.0,2424.5,1,10.0,312,245.54


In [51]:
# convert date column to date tyoe data
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

In [52]:
#df['Coupon_Code'] = pd.factorize(df['Coupon_Code'])[0]
df['Product_Category'] = pd.factorize(df['Product_Category'])[0]
df['Gender'] = pd.factorize(df['Gender'])[0]
df['Location'] = pd.factorize(df['Location'])[0]
df['Coupon_Status'] = pd.factorize(df['Coupon_Status'])[0]
df.head(2)

,CustomerID,Gender,Location,Tenure_Months,Transaction_Date,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,GST,Offline_Spend,Online_Spend,Month,Discount_pct,Produc_ID,score
0,17850.0,0,0,12.0,2019-01-01,0,1.0,153.71,6.5,0,0.1,4500.0,2424.5,1,10.0,316,153.71
2,17850.0,0,0,12.0,2019-01-01,0,2.0,122.77,6.5,1,0.1,4500.0,2424.5,1,10.0,312,245.54


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48672 entries, 0 to 52923
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   CustomerID        48672 non-null  float64       
 1   Gender            48672 non-null  int64         
 2   Location          48672 non-null  int64         
 3   Tenure_Months     48672 non-null  float64       
 4   Transaction_Date  48672 non-null  datetime64[ns]
 5   Product_Category  48672 non-null  int64         
 6   Quantity          48672 non-null  float64       
 7   Avg_Price         48672 non-null  float64       
 8   Delivery_Charges  48672 non-null  float64       
 9   Coupon_Status     48672 non-null  int64         
 10  GST               48672 non-null  float64       
 11  Offline_Spend     48672 non-null  float64       
 12  Online_Spend      48672 non-null  float64       
 13  Month             48672 non-null  int64         
 14  Discount_pct      48672 non

# building and testing the recommender system

In [54]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
print(f"Training users: {len(train_df['CustomerID'].unique())}")
print(f"Test users: {len(test_df['CustomerID'].unique())}")
#print(f"user in encoder: {len(user_encoder.classes_)}")

Training users: 1450
Test users: 1303


In [55]:
import random

augmented = train_df.copy()

augmented['Avg_Price'] = augmented['Avg_Price'] + [random.uniform(0.95, 1.05) for _ in range(len(augmented))]
augmented['Quantity'] = [max(1, q + random.choice([-1,0,1])) for q in augmented['Quantity']]
augmented['score'] = augmented['Avg_Price'] * augmented['Quantity']

train_df = pd.concat([train_df, augmented], ignore_index=True)
print(f"Training: {len(train_df)}")

Training: 77874


# hybrid recommender using, popularity,  content based + collaborative filtering

In [56]:
# hybrid model: popularity, personalization (content based) + collaborative filtering
test_users = test_df['CustomerID'].unique()
hybrid_precisions, hybrid_recalls = [], []

In [57]:
# define the weights
pop_weight = 1/3
personal_weight = 1/3
collab_weight = 1/3

In [58]:
# popularity baseline
popular_items = train_df.groupby('Produc_ID')['score'].sum().nlargest(15).index.tolist()
pop_count = int(15 * pop_weight)

In [59]:
# collabrative filtering
user_item_matrix = train_df.pivot_table(index='CustomerID', columns='Produc_ID', values='score', fill_value=0)
collab_count = int(15 * collab_weight)

In [60]:
# content based
for user in test_users:
    true_items = test_df[test_df['CustomerID'] == user]['Produc_ID'].tolist()
    if not true_items: continue
    recommendations = []

    # part 1: popularity
    recommendations.extend(popular_items[:pop_count])
    # part 2: collbrative filtering
    if user in user_item_matrix.index:
        # finding similar usiers
        from sklearn.metrics.pairwise import cosine_similarity
        user_vector = user_item_matrix.loc[user].values.reshape(1,-1)
        similarities = cosine_similarity(user_vector, user_item_matrix.values)[0]
        # get the top similar users
        similar_users_idx = np.argsort(similarities)[-6:-1]
        similar_users = user_item_matrix.iloc[similar_users_idx]
        # get the items like by the simialr users
        user_interacted = set(train_df[train_df['CustomerID'] == user]['Produc_ID'])
        similar_users_items = similar_users.sum().sort_values(ascending=False)
        collab_items = [item for item in similar_users_items.index
                        if item not in user_interacted and item not in recommendations]
        recommendations.extend(collab_items[:collab_count])

    # part 3: content based
    if user in train_df['CustomerID'].values:
        user_items = train_df[train_df['CustomerID'] == user]['Produc_ID'].unique()
        user_categories = train_df[train_df['CustomerID'] == user]['Product_Category'].value_counts()
        if len(user_categories) > 0:
            top_category = user_categories.index[0]
            category_items = train_df[train_df['Product_Category'] == top_category]
            cat_popular = category_items.groupby('Produc_ID')['score'].sum().nlargest(10).index.tolist()
            # add categry items not in reccomendations
            personal_items = [item for item in cat_popular
                            if item not in recommendations and item not in user_items]
            personal_count = 15 - len(recommendations)
            recommendations.extend(personal_items[:personal_count])
    # make sure we have exactly 15 item and remove duplicates
    recommendations = list(dict.fromkeys(recommendations))[:15]
    # full if neded the remianig slot swith popularity
    if len(recommendations) < 15:
        additional_popular = [item for item in popular_items if item not in recommendations]
        recommendations.extend(additional_popular[:15 - len(recommendations)])
    hits = len(set(recommendations) & set(true_items))
    hybrid_precisions.append(hits / len(recommendations))
    hybrid_recalls.append(hits / len(true_items))

In [61]:
# the results
print(" Hybrid recommender ")
print(f"Precision@15: {np.mean(hybrid_precisions):.4f}")
print(f"Recall@15: {np.mean(hybrid_recalls):.4f}")
print(f"weights - popularity: {pop_weight:.2f}, collabrative filtering: {collab_weight:.2f}, Personalization: {personal_weight:.2f}")

 Hybrid recommender 
Precision@15: 0.1086
Recall@15: 0.2780
weights - popularity: 0.33, collabrative filtering: 0.33, Personalization: 0.33


Dataset reference:
https://www.kaggle.com/datasets/jacksondivakarr/online-shopping-dataset (20/11/2025)